This notebook explodes the order items array into one row per item and creates the silver orders table.

In [0]:
df_orders = spark.read.table('gizmobox.silver.py_orders_json')
display(df_orders)

In [0]:
from pyspark.sql.functions import *

In [0]:
df_normalised_orders = df_orders.select(

'json_value.customer_id',
'json_value.items',
'json_value.order_id',
'json_value.order_status',
'json_value.payment_method',
'json_value.total_amount',
'json_value.transaction_timestamp' 

)
display(df_normalised_orders)

In [0]:
df_normalised_orders = df_orders.select(
'json_value.customer_id',
'json_value.order_id',
'json_value.order_status',
'json_value.payment_method',
'json_value.total_amount',
'json_value.transaction_timestamp',
array_distinct('json_value.items').alias('items') )

display(df_normalised_orders)

In [0]:
df_explode_orders = df_normalised_orders.select(
'customer_id',
'order_id',
'order_status',
'payment_method',
'total_amount',
'transaction_timestamp',
explode('items').alias('items') )

display(df_explode_orders)

In [0]:
df_order_items = (
    df_explode_orders
    .select(
        'order_id',
'order_status',
'payment_method',
'total_amount',
'transaction_timestamp',
'customer_id',
'items.item_id',
'items.name',
'items.price',
'items.quantity',
'items.category',
'items.details.brand',
'items.details.color'
    )
)
display(df_order_items)

In [0]:
df_order_items.write.mode('overwrite').saveAsTable('gizmobox.silver.py_orders_items')